##### APs and slits:  SLH_D1316 (optional), AP1408, AP1420 will be inserted. Take out later as needed
##### will steer with H1305,  HV1345,  HV1385
##### will maximize FC1420
##### Check FaradayCup Average -> turn off and later turn back on after optimization

In [1]:
import time
import datetime
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import OrderedDict
from epics import caput, caget, caget_many

In [2]:
import sys
repo_root = '/projects/rea3/AP-ReA/jupyter-ML/pkgs/stBO'
sys.path.insert(0, str(repo_root))
from stbo.optimization import BOController
from stbo.utils import live_monitor_plot, live_history_plot

In [3]:
repo_root= '/projects/rea3/AP-ReA/jupyter-ML/pkgs/machineIO'
sys.path.insert(0, str(repo_root))
from machineIO import construct_machineIO, Evaluator, OracleEvaluator
from machineIO.objFunc import SingleTaskObjectiveFunction
from machineIO import preset

# === Begin User Inputs ===
###### ignore any variable name that starts with "_"

In [4]:
#=== Faraday-Cup PV and goal =========================
FC='REA_BTS43:MTER_N0001:I_RD'

FCgoal = 140e-12


#=== IO settings =====================================
timespan_for_average = 4.0  # [sec]
additional_wait_after_powersupply_ramp  = 0.5 # [sec]
_expected_oracle_time_cost = timespan_for_average + additional_wait_after_powersupply_ramp + 1


#=== Optimizer setting ===============================
is_close_to_opt = True  # True for local optimization. 


#=== Control knobs ===================================
control_CSETs= [
    "REA_CB01:DCH_D1342:I_CSET",
    "REA_CB01:DCV_D1342:I_CSET",
    "REA_CB01:DCH_D1362:I_CSET",
    "REA_CB01:DCV_D1362:I_CSET",
]

is_cryo = np.any(['_CB' in pv or '_CA' in PV for pv in control_CSETs])
if not is_close_to_opt and is_cryo:
    print('warn: global optimization is proposed with correctors in croyo module. proceed with care')

#### Optimizer setting

In [5]:
if is_close_to_opt:
    n_init       = 10   # at least number of control dim
    n_global     = 0
    n_local      = 30
    n_finetune   = 5    # about number of control dim
else:
    n_init       = 25 
    n_global     = 25
    n_local      = 25
    n_finetune   = 5
    
_budget = n_init +n_global +n_local +n_finetune
print(f"budget: {_budget}")
print(f"expected run time: {int(_budget*_expected_oracle_time_cost)} sec")

budget: 45
expected run time: 247 sec


# === End User Inputs ===

### IO setting

In [6]:
_data_acquire_rate = 5 # [Hz]
_expected_bo_computation_time = 1
_run_async = 0.5 < abs(_expected_oracle_time_cost/_expected_bo_computation_time - 1) < 2
             
print(f"_run_async: {_run_async}")

io = construct_machineIO(
    fetch_data_time_span=timespan_for_average,
    ensure_set_timewait_after_ramp=additional_wait_after_powersupply_ramp,
    sample_interval = 1/_data_acquire_rate,
    test = False,
)

_run_async: False


# prepare beam and machine status

##### get beam info

In [7]:
ion = caget("REA_EXP:ELMT")
Q = int(caget("REA_EXP:Q"))
A = int(caget("REA_EXP:A"))
AQ = A/Q
ion = str(A)+ion+str(Q)
print(ion, 'A/Q=',AQ)

14N6 A/Q= 2.3333333333333335


##### Prepare Apertures

In [8]:
aperture_setPVs = ["REA_BTS41:AP_D1393:IN_CMD","REA_BTS43:AP_D1430:IN_CMD"]
aperture_rdPVs  = ["REA_BTS41:AP_D1393:IN_CMD","REA_BTS43:AP_D1430:IN_CMD"]
aperture_rd_targets = [1.0,1.0]
aperture_rd_tols    = [0.1,0.1]


for i,pv in enumerate(aperture_rdPVs):
    pv_simple = pv.strip(':POS_RD').strip(':IN_CMD')
    target = aperture_rd_targets[i]
    val = caget(pv)
    tol = aperture_rd_tols[i]
    isin = target-tol < val < target+tol
    if not isin:
        r = input(f"aperture {pv_simple} is not inserted. Do you allow me to put it in? ")
        if r in ['yes','y','YES','Y']:
            caput(aperture_setPVs[i],target)
            time.sleep(1)
            print(f"aperture {pv_simple} is inserted")
            
time.sleep(5)
for i,pv in enumerate(aperture_rdPVs):
    pv_simple = pv.strip(':POS_RD').strip(':IN_CMD')
    target = aperture_rd_targets[i]
    val = caget(pv)
    tol = aperture_rd_tols[i]
    isin = target-tol < val < target+tol
    if not isin:
        r = input(f"aperture {pv_simple} is still not inserted. please check manually. enter to continue")

##### Turn off correctors b/w Apertures

In [9]:
corrector_setPVs = ["REA_BTS42:DCH_D1397:I_CSET","REA_BTS42:DCV_D1397:I_CSET"]
corrector_rdPVs  = ["REA_BTS42:DCH_D1397:I_RD",  "REA_BTS42:DCV_D1397:I_RD"]
corrector_rd_targets = [0]*len(corrector_setPVs)
corrector_rd_tols    = [0.01,0.01]


for i,pv in enumerate(corrector_setPVs):
    pv_simple = pv.strip(':I_CSET')
    target = corrector_rd_targets[i]
    val = caget(pv)
    tol = corrector_rd_tols[i]
    isin = target-tol < val < target+tol
    if not isin:
        r = input(f"corrector {pv_simple} is not off. Do you allow me to set it to zero? ")
        if r in ['yes','y','YES','Y']:
            caput(corrector_setPVs[i],target)
            time.sleep(1)
            print(f"corrector {pv_simple} is set to zero")
            
time.sleep(5)
for i,pv in enumerate(corrector_setPVs):
    pv_simple = pv.strip(':I_CSET')
    target = corrector_rd_targets[i]
    val = caget(pv)
    tol = corrector_rd_tols[i]
    isin = target-tol < val < target+tol
    if not isin:
        r = input(f"corrector {pv_simple} is still not off. please check manually. enter to continue")

corrector REA_BTS42:DCH_D1397 is not off. Do you allow me to set it to zero? y
corrector REA_BTS42:DCH_D1397 is set to zero
corrector REA_BTS42:DCV_D1397 is not off. Do you allow me to set it to zero? y
corrector REA_BTS42:DCV_D1397 is set to zero


##### Check upstream FCs

In [10]:
upstream_FC_isoutPVs  = ["REA_BTS09:FC_D0864:LMOUT_RSTS", 
                         "REA_BTS10:FCS_D0918:LMOUT_RSTS", 
                         "REA_BTS10:FCS_D0947:LMOUT_RSTS", 
                         "REA_BTS19:FC_D0977:LMOUT_RSTS", 
                         "REA_BTS19:FC_D0999:LMOUT_RSTS", 
                         "REA_WK01:FC_D1058:LMOUT_RSTS", 
                         "REA_WL01:FC_D1096:LMOUT_RSTS", 
                         "REA_WM01:FC_D1148:LMOUT_RSTS", 
                         "REA_BTS25:FC_D1178:LMOUT_RSTS", 
                         "REA_BTS25:FC_D1207:LMOUT_RSTS", 
                         "REA_BTS30:FC_D1256:LMOUT_RSTS", 
                         "REA_BTS40:FC_D1306:LMOUT_RSTS", 
                         "REA_BTS41:FC_D1393:LMOUT_RSTS"]
upstream_FC_insertPVs = [pv.replace("LMOUT_RSTS","IN_CMD") for pv in upstream_FC_isoutPVs]

for FC_isoutPV, FC_insertPV in zip(upstream_FC_isoutPVs, upstream_FC_insertPVs):
    is_FC_out = caget(FC_isoutPV)
    if not is_FC_out:
        r = input(f"FC {FC_insertPV.strip(':IN_CMD')} is inserted. Do you allow me to take it out? ")
        if r in ['yes','y','YES','Y']:
            caput(FC_insertPV,0)
            time.sleep(1)
time.sleep(5)
for FC_isoutPV, FC_insertPV in zip(upstream_FC_isoutPVs, upstream_FC_insertPVs):
    is_FC_out = caget(FC_isoutPV)
    if not is_FC_out:
        r = input(f"FC {FC_insertPV.strip(':IN_CMD')} is still inserted. please check manually. enter to continue")

##### prepare FC

In [11]:
FC_isout_PV = "REA_BTS43:FC_D1432:LMOUT_RSTS"
FC_insertPV = "REA_BTS43:FC_D1432:IN_CMD"

is_FC_out = caget(FC_isout_PV)
if is_FC_out:
    r = input(f"FC {FC_insertPV.strip(':IN_CMD')} is not inserted. Do you allow me to put it in? ")
    if r in ['yes','y','YES','Y']:
        caput(FC_insertPV,1)
        time.sleep(5)
current_FC_read = caget(FC)
print(f"{FC} reads: {current_FC_read}")
assert current_FC_read < FCgoal
if not current_FC_read > 0.05*FCgoal:
    input("FC reading too small. Check if any upstream FC is inserted")

is_FC_averaging = caget("REA_BTS43:MTER_N0001:AVG_RSTS") > 0
if is_FC_averaging:
    r = input(f"REA_BTS43:MTER_N0001:AVG_RSTS is averaging. Do you allow me to turn off averaging? ")
    if r in ['yes','y','YES','Y']:
        caput("REA_BTS43:MTER_N0001:AVG_CMD",0)
        time.sleep(1)

REA_BTS43:MTER_N0001:I_RD reads: 3.305836e-11


# Define control bounds

In [12]:
x0 = caget_many(control_CSETs)

control_RDs  = [pv.replace('_CSET','_RD') for pv in control_CSETs]
control_Lo_limit, control_Hi_limit = preset.get_limits(control_CSETs)
control_limit_half_size = 0.5*(control_Hi_limit - control_Lo_limit)
control_bound_half_size = AQ*0.1*control_limit_half_size
control_mid  = x0 # np.array([0 if ':DC' in pv else v for v,pv in zip(x0,control_CSETs)])
control_tols = 0.01*control_limit_half_size
control_min  = control_mid - control_bound_half_size
control_max  = control_mid + control_bound_half_size
control_min = np.clip(control_min, a_min = control_Lo_limit, a_max = None)
control_max = np.clip(control_max, a_min = None, a_max = control_Hi_limit)
assert np.all(control_min<x0)
assert np.all(x0<control_max)

oracle_key_names = {
    'x':control_RDs,
    'y':'composite_obj'
}
control_bounds = torch.tensor([control_min.tolist(), control_max.tolist()], dtype=torch.float64)

print("============== check control bounds ================= ")
pd.DataFrame(np.array([x0,control_min,control_max,control_tols,control_Lo_limit,control_Hi_limit]).T,
             index=control_CSETs, 
             columns=['current value','control min','control max','tol','LoLim','HiLim'])

============== check control bounds ================= 


,current value,control min,control max,tol,LoLim,HiLim
REA_CB01:DCH_D1342:I_CSET,-0.101287,-2.434620,2.232046,0.1,-10.0,10.0
REA_CB01:DCV_D1342:I_CSET,0.744140,-1.589194,3.077473,0.1,-10.0,10.0
REA_CB01:DCH_D1362:I_CSET,-0.308219,-2.641552,2.025114,0.1,-10.0,10.0
REA_CB01:DCV_D1362:I_CSET,-0.498461,-2.831794,1.834872,0.1,-10.0,10.0


# Define objectives

In [13]:
objective_goal      = {FC: {'more than': FCgoal}}
objective_tolerance = {FC: 0.2*FCgoal}
objective_weight    = {FC: 1.0}
monitor_PVs = [FC]

##== Display objective info


============== check objective ================= 
 too small numbers may rounded to show 0 but actual value may not be 0


,goal,norm,weight
REA_BTS43:MTER_N0001:I_RD,{'more than': 1.4e-10},0.0,1.0


# setup and test evaluator and oracle

In [14]:
obj_func = SingleTaskObjectiveFunction(
    objective_PVs = list(objective_goal.keys()), 
    composite_objective_name = 'composite_obj',
    objective_goal = objective_goal,
    objective_weight = objective_weight,
    objective_tolerance = objective_tolerance,
    p_order = 1,
    apply_bilog = False
)

In [15]:
oracle =  OracleEvaluator(
    io,
    control_CSETs= control_CSETs,
    control_RDs  = control_RDs,
    control_tols = control_tols,
    oracle_key_names = oracle_key_names,
    monitor_PVs  = monitor_PVs,
    df_manipulators = [obj_func.calculate_objectives_from_df],
)

# run BO

In [16]:
now0 = datetime.datetime.now()
fname = now0.strftime('%Y%m%d_%H%M')+'['+ion+'][REA][simBO]FC1316'
fname

'20260417_1416[14N6][REA][simBO]FC1316'

In [17]:
bo = BOController(oracle, 
                  bounds = control_bounds)

[14:16:25.849] WARNING: phantasy.~.epics_tools: Established 9 PVs in 20.8 ms.


###### preprare live plot  --> specify PV groups (list of list) to plot togather

In [18]:
monitor_groups = [[FC]]
control_CSETs_groups = [control_CSETs]
control_RDs_groups = [control_RDs]

monitor_live = live_monitor_plot(
    bo, oracle,
    monitor_groups       = monitor_groups, 
    control_CSETs_groups = control_CSETs_groups,
    control_RDs_groups   = control_RDs_groups,
)
history_live = live_history_plot(bo)

monitor_live.start()
history_live.start()

In [19]:
bo.initialize(budget=n_init, local_init=is_close_to_opt)

[14:16:31.221] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.2 ms.
[14:16:37.192] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[14:16:43.204] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[14:16:49.201] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[14:16:55.027] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.2 ms.
[14:17:01.014] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[14:17:06.991] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[14:17:13.016] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[14:17:19.392] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.


In [20]:
for _ in range(n_global):
    fresh_train = True #(not bool(_%2)) or len(bo.train_x) < bo.bounds.shape[1]*2
    bo.step(mode="global", acq_type="qEI", fresh_train=fresh_train, plot_acq=False, asynchro=_run_async)
    plt.show()

In [21]:
for _ in range(n_local):
    fresh_train = True #(not bool(_%2)) or len(bo.train_x) < bo.bounds.shape[1]*2
    bo.step(mode="local", acq_type="qEI", fresh_train=fresh_train, plot_acq=False, asynchro=_run_async)
    plt.show()

[14:17:24.990] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[14:17:33.236] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[14:17:40.197] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[14:17:49.026] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[14:17:56.597] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[14:18:05.402] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[14:18:13.821] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[14:18:20.190] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[14:18:25.794] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[14:18:31.599] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[14:18:37.595] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.2 ms.
[14:18:43.790] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[14:18:48.645] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.

In [22]:
for _ in range(n_finetune):
    bo.step(mode="fine_tune", acq_type="qEI", fresh_train=True, plot_acq=False, asynchro=_run_async)
    plt.show()

[14:20:30.791] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[14:20:36.807] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[14:20:42.591] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[14:20:47.610] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.2 ms.
[14:20:53.214] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.


In [23]:
now1 = datetime.datetime.now()
print(f"optimization took {(now1-now0).seconds} sec")

optimization took 271 sec


# set best solution

In [24]:
x_hist = [h["x"] for h in bo.history]
y_hist = [h["y"] for h in bo.history]
imax = np.argmax(y_hist)

orcle_dic = oracle(x_hist[imax])

print("Best x:", x_hist[imax])
print("Best y (old):", y_hist[imax])
print("Best y (new):", orcle_dic['y'])

[14:20:58.993] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
Best x: [-0.09484335  0.50540127 -0.13323915 -0.37914316]
Best y (old): [-3.30391645]
Best y (new): [-3.38625121]


In [25]:
tmp = np.vstack((x0,x_hist[imax]))
pd.DataFrame(tmp,columns=control_CSETs,index=['before opt','after opt']).T

,before opt,after opt
REA_CB01:DCH_D1342:I_CSET,-0.101287,-0.094843
REA_CB01:DCV_D1342:I_CSET,0.744140,0.505401
REA_CB01:DCH_D1362:I_CSET,-0.308219,-0.133239
REA_CB01:DCV_D1362:I_CSET,-0.498461,-0.379143


[14:21:19.015] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.3 ms.


# plot model
not yet implemented